# Libraries

In [ ]:
import pandas as pd
import json
from sklearn.neighbors import KNeighborsClassifier
import re
from sentence_transformers import SentenceTransformer, util

# Read Data

In [ ]:
file = pd.read_json("Reservations.json")

In [ ]:
file.columns.to_list()

# Handeling nested JSON using Flattening

## products

In [ ]:
# Function to flatten one product (first only, or adjust to handle multiple)
def extract_product_info(row):
    if isinstance(row, dict) and 'data' in row and isinstance(row['data'], list) and len(row['data']) > 0:
        product = row['data'][0]  # take the first product
        flat = {}

        # Basic fields
        keys_to_extract = ['ref', 'type', 'title', 'to_date', 'to_time', 'from_date', 'from_time', 
                           'description', 'product_sku', 'product_code', 'pu_participants']
        for key in keys_to_extract:
            flat[f'products_{key}'] = product.get(key)

        # Nested: product_data
        if 'product_data' in product and 'data' in product['product_data']:
            pdata = product['product_data']['data']
            for key in ['mealplan_code', 'rateplan_code', 'rateplan_title']:
                flat[f'products_product_data_{key}'] = pdata.get(key)

        # Nested: pu_participants_details
        if 'pu_participants_details' in product and 'data' in product['pu_participants_details']:
            participants = product['pu_participants_details']['data']
            age_counts = {}
            for p in participants:
                age_code = p.get('age_code')
                if age_code:
                    age_counts[age_code] = age_counts.get(age_code, 0) + 1
            for code, count in age_counts.items():
                flat[f'products_participants_{code}_count'] = count

        return flat
    return {}

# Apply to DataFrame
products_expanded = file['products'].apply(extract_product_info).apply(pd.Series)

# Merge back
file = pd.concat([file, products_expanded], axis=1)

## offer_variants

In [ ]:
def extract_offer_variant_info(row):
    flat = {}

    # Ensure expected format
    if isinstance(row, dict) and 'data' in row and isinstance(row['data'], list) and len(row['data']) > 0:
        first_offer = row['data'][0]

        # total participants in the offer
        flat['offer_variants_total_participants'] = first_offer.get('total_participants')

        # product inside the offer
        if 'products' in first_offer and 'data' in first_offer['products']:
            products = first_offer['products']['data']
            if isinstance(products, list) and len(products) > 0:
                product = products[0]

                # Basic product fields
                keys_to_extract = ['ref', 'type', 'title', 'to_date', 'to_time', 'from_date', 'from_time', 
                                   'description', 'product_sku', 'product_code', 'pu_participants']
                for key in keys_to_extract:
                    flat[f'offer_variants_products_{key}'] = product.get(key)

                # Nested product_data
                if 'product_data' in product and 'data' in product['product_data']:
                    pdata = product['product_data']['data']
                    for key in ['mealplan_code', 'rateplan_code', 'rateplan_title']:
                        flat[f'offer_variants_products_product_data_{key}'] = pdata.get(key)

                # Nested pu_participants_details
                if 'pu_participants_details' in product and 'data' in product['pu_participants_details']:
                    participants = product['pu_participants_details']['data']
                    age_counts = {}
                    for p in participants:
                        age_code = p.get('age_code')
                        if age_code:
                            age_counts[age_code] = age_counts.get(age_code, 0) + 1
                    for code, count in age_counts.items():
                        flat[f'offer_variants_products_participants_{code}_count'] = count

    return flat

# Apply to the DataFrame
offer_variants_expanded = file['offer_variants'].apply(extract_offer_variant_info).apply(pd.Series)

# Merge into original DataFrame
file = pd.concat([file, offer_variants_expanded], axis=1)

## customer

In [ ]:
def extract_customer_info(row):
    flat = {}

    if isinstance(row, dict) and 'data' in row and isinstance(row['data'], dict):
        data = row['data']

        # All flat fields
        flat_keys = [
            'id', 'account_id', 'partner_code', 'partner_id', 'deleted_at', 'language', 'type', 'main', 'vip',
            'gender', 'salutation', 'name_prefix', 'firstname', 'lastname', 'companyname', 'birthday',
            'birth_municipality_code', 'birth_country_iso', 'nationality_iso', 'blacklist',
            'average_stay_time', 'stay_count', 'total_revenue', 'registration', 'last_contact', 'last_stay',
            'last_reservation', 'last_inquiry', 'legitimate_interest_till', 'legitimate_interest_lastmod',
            'lastmod', 'delete_on', 'delete_on_lastmod'
        ]
        for key in flat_keys:
            flat[f'customer_{key}'] = data.get(key)

        # partner_created_by
        partner_created = data.get('partner_created_by') or {}
        flat['customer_partner_created_by_username'] = partner_created.get('username')
        flat['customer_partner_created_by_label'] = partner_created.get('label')

        # correspondence flags
        correspondence = data.get('correspondence') or {}
        if isinstance(correspondence, dict):
            for key, val in correspondence.items():
                flat[f'customer_correspondence_{key}'] = val

        # address (first primary)
        addresses = data.get('addresses', {}).get('data', [])
        for addr in addresses:
            if addr.get('primary'):
                for key in ['street', 'city', 'zip', 'country_iso', 'firstname', 'lastname', 'companyname']:
                    flat[f'customer_address_{key}'] = addr.get(key)
                flat['customer_address_newsletter_status'] = addr.get('newsletter_status')
                flat['customer_address_newsletter_status_lastmod'] = addr.get('newsletter_status_lastmod')
                break

        # email (first primary)
        emails = data.get('emails', {}).get('data', [])
        for email in emails:
            if email.get('primary'):
                flat['customer_email_address'] = email.get('address')
                flat['customer_email_newsletter_status'] = email.get('newsletter_status')
                flat['customer_email_newsletter_status_lastmod'] = email.get('newsletter_status_lastmod')
                break

        # phone (first primary phone or mobile)
        phones = data.get('phones', {}).get('data', [])
        for phone in phones:
            if phone.get('primary'):
                phone_type = phone.get('type')
                flat[f'customer_phone_{phone_type}'] = phone.get('phone_nr')
                flat[f'customer_phone_{phone_type}_newsletter_status'] = phone.get('newsletter_status')
                flat[f'customer_phone_{phone_type}_newsletter_status_lastmod'] = phone.get('newsletter_status_lastmod')
                break

        # profiling_tags: count + names
        tags = data.get('profiling_tags', {}).get('data', [])
        flat['customer_profiling_tag_count'] = len(tags)
        tag_names = [t.get('name') for t in tags if t.get('name')]
        flat['customer_profiling_tag_names'] = ', '.join(tag_names)

    return flat

# Apply and flatten to new DataFrame
customer_expanded = file['customer'].apply(extract_customer_info).apply(pd.Series)

# Merge with the original DataFrame
file = pd.concat([file, customer_expanded], axis=1)

## advertising

In [ ]:
def extract_advertising_info(row):
    if isinstance(row, dict):
        return {
            'advertising_source': row.get('source'),
            'advertising_campaign': row.get('campaign')
        }
    return {}

# Apply to DataFrame
advertising_expanded = file['advertising'].apply(extract_advertising_info).apply(pd.Series)

# Merge into the main DataFrame
file = pd.concat([file, advertising_expanded], axis=1)

## total_price_details

In [ ]:
def extract_total_price_details(row):
    flat = {}
    if isinstance(row, dict) and 'data' in row and isinstance(row['data'], list) and len(row['data']) > 0:
        item = row['data'][0]
        # Extract all meaningful fields
        keys_to_extract = ['title', 'amount', 'pu_price', 'quantity', 'product_ref']
        for key in keys_to_extract:
            flat[f'total_price_details_{key}'] = item.get(key)
    return flat

# Apply and convert to DataFrame
total_price_details_expanded = file['total_price_details'].apply(extract_total_price_details).apply(pd.Series)

# Merge into original DataFrame
file = pd.concat([file, total_price_details_expanded], axis=1)

## First Copy

### Save a copy

In [ ]:
def save_dataframe(df, data_path='data.json', schema_path='schema.json'):
    # Save the data
    df.to_json(data_path, orient="records", indent=2, date_format='iso', force_ascii=False)
    
    # Save the schema (data types)
    schema = {col: str(dtype) for col, dtype in df.dtypes.items()}
    with open(schema_path, "w") as f:
        json.dump(schema, f, indent=2)

save_dataframe(file, data_path="Figure_out_data_withoutEdit.json", schema_path="Figure_out_data_schema_withoutEdit.json")

### Load the copy

In [ ]:
def load_dataframe(data_path='data.json', schema_path='schema.json'):
    # Load data
    df = pd.read_json(data_path, orient="records")
    
    # Load schema
    with open(schema_path) as f:
        schema = json.load(f)
    
    # Identify column types
    datetime_cols = [col for col, dtype in schema.items() if 'datetime64' in dtype]
    category_cols = [col for col, dtype in schema.items() if dtype == 'category']
    
    # Reparse datetimes
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], utc='UTC' in schema[col])
    
    # Restore categories
    for col in category_cols:
        df[col] = df[col].astype('category')
    
    return df

file = load_dataframe(data_path="Figure_out_data_withoutEdit.json", schema_path="Figure_out_data_schema_withoutEdit.json")

# Data preprocessing

## Data Cleaning & Feature Selection

### Drop the nested JSON columns

In [ ]:
nested_container_cols = [
    "products",
    "offer_variants",
    "customer",
    "advertising",
    "total_price_details"
]

file = file.drop(columns=nested_container_cols)

### Drop columns that are 100% missing

In [ ]:
# Assuming your dataframe is named: file
df = file

# 1) Missing rate per column
missing_rate = df.isna().mean()

# 2) Columns that are 100% missing
cols_all_missing = missing_rate[missing_rate == 1.0].index.tolist()

# Print results
print(f"Number of columns with 100% missing values: {len(cols_all_missing)}")

if cols_all_missing:
    print("\nColumns with 100% missing values:")
    for col in cols_all_missing:
        print(f" - {col}")
else:
    print("\nNo columns with 100% missing values found.")

In [ ]:
# 3) Drop them
file = file.drop(columns=cols_all_missing)

### Drop columns that are 95% or more missing

In [ ]:
# Assuming your dataframe is named: file
df = file

# 1) Missing rate per column
missing_rate = df.isna().mean()

# 2) Columns that are 100% missing
cols_all_missing = missing_rate[missing_rate >= 0.95].index.tolist()

# Print results
print(f"Number of columns with 95% or more missing values: {len(cols_all_missing)}")

if cols_all_missing:
    print("\nColumns with 95% or more missing values:")
    for col in cols_all_missing:
        print(f" - {col}")
else:
    print("\nNo columns with 100% or more missing values found.")

In [ ]:
# Column to keep despite high missingness
keep_sparse_cols = [
    "offer_variants_products_participants_infant_count"
]
# Final drop list: >=95% missing EXCEPT kept columns
cols_to_drop = [
    c for c in cols_all_missing if c not in keep_sparse_cols
]
# Drop them
file = file.drop(columns=cols_to_drop)

### Drop constant columns 

In [ ]:
col = "business_customer"

print("dtype:", file[col].dtype)
print("python types (top 5):")
print(file[col].map(lambda x: type(x).__name__).value_counts().head(5))

print("\nSample non-null values (first 5):")
print(file[col].dropna().head(5).tolist())

In [ ]:
df = file  # or df_step1 if you've already done Step 1

def to_hashable(x):
    """Convert unhashable types (dict/list/set) into a stable representation."""
    if isinstance(x, dict):
        # stable order for keys
        return str(sorted(x.items()))
    if isinstance(x, (list, set, tuple)):
        return str(list(x))
    return x

cols_constant = []

for col in df.columns:
    s = df[col]

    # Convert only if needed (object columns are most likely to contain dict/list)
    if s.dtype == "object":
        s2 = s.map(to_hashable)
    else:
        s2 = s

    # number of unique non-null values
    n_unique = s2.dropna().nunique()

    if n_unique <= 1:
        cols_constant.append(col)

print(f"Number of constant columns: {len(cols_constant)}")

if cols_constant:
    print("\nConstant columns (single unique value):")
    for col in cols_constant:
        print(f" - {col}")
else:
    print("\nNo constant columns found.")

In [ ]:
file = file.drop(columns=cols_constant)

### Dealing with customer* columns for GDPR review and drop 

In [ ]:
df = file

# Find columns containing 'customer' (case-insensitive)
customer_cols = [col for col in df.columns if 'customer' in col.lower()]

print(f"Number of customer-related columns: {len(customer_cols)}")

print("\nCustomer-related columns:")
for col in sorted(customer_cols):
    print(f" - {col}")

### Drop GDPR-restricted customer columns

In [ ]:
gdpr_drop_cols = [
    "business_customer",
    "customer_salutation",
    "customer_firstname",
    "customer_lastname",
    "customer_address_street",
    "customer_address_zip",
    "customer_address_firstname",
    "customer_address_lastname",
    "customer_email_address",
    "customer_phone_phone",
    "customer_phone_mobile",
    "customer_legitimate_interest_till",
    "customer_legitimate_interest_lastmod",
    "customer_lastmod",
    "customer_delete_on",
    "customer_delete_on_lastmod",
    "customer_address_newsletter_status",
    "customer_address_newsletter_status_lastmod",
    "customer_email_newsletter_status",
    "customer_email_newsletter_status_lastmod",
    "customer_phone_phone_newsletter_status",
    "customer_phone_phone_newsletter_status_lastmod",
    "customer_profiling_tag_count",
    "customer_profiling_tag_names",
    "customer_correspondence_disable_feedback",
    "customer_correspondence_disable_greeting",
    "customer_correspondence_disable_reminder",
    "customer_correspondence_disable_pre_experience",
    "customer_correspondence_disable_post_experience",
    "customer_name_prefix"
]
file = file.drop(columns=gdpr_drop_cols)

### Drop internal system timestamps (not customer behavior) and pure identifiers & internal system keys

In [ ]:
is_drop_cols = [
    "created_at",
    "updated_at",
    "partner_created_at",
    "partner_modified_at",
    "id",
    "partner_sale_id",
    "partner_sale_slug",
    "origin_sale_id"
]
file = file.drop(columns=is_drop_cols)

### Drop requested_at

In [ ]:
file = file.drop(columns=['requested_at'])

### Drop free-text & unstructured fields

In [ ]:
ft_drop_cols = [
    "note",
    "description",
    "offer_variants_products_description"
]
file = file.drop(columns=ft_drop_cols)

### Drop redundant product identifiers

In [ ]:
rpi_drop_cols = [
    "products_product_sku",
    "products_product_code",
    "offer_variants_products_product_sku",
    "offer_variants_products_product_code"
]
file = file.drop(columns=rpi_drop_cols)

In [ ]:
file.columns.tolist()

### Drop different columns for different reason

In [ ]:
def_drop_cols = [
    "pos",
    "language",
    "products_ref",
    "total_price_details_product_ref",
    "products_from_time",
    "products_to_time",
    "offer_variants_products_from_time",
    "offer_variants_products_to_time",
    "customer_partner_id",
    "customer_type",
    "title",
    "total_participants"
]
file = file.drop(columns=def_drop_cols)

### Restricting the Dataset to Realized Orders and drop the column type

In [ ]:
file = file[file["type"] == "order"].copy()

In [ ]:
file = file.drop(columns=['type'])

### Drop missing customer_id rows

In [ ]:
file = file[file["customer_id"].notna()].copy()

### Deal with missing values through all columns

In [ ]:
missing_summary = (
    pd.DataFrame({
        "column": file.columns,
        "dtype": file.dtypes.values,
        "missing_count": file.isna().sum().values,
        "missing_percent": (file.isna().mean() * 100).round(2).values,
        "unique_non_null": file.nunique(dropna=True).values
    })
    .sort_values("missing_percent", ascending=False)
    .reset_index(drop=True)
)

missing_summary

### Drop offer_* columns 

In [ ]:
offer_drop_cols = [
    "offer_variants_products_participants_infant_count",
    "offer_variants_products_participants_child_count",
    "offer_variants_products_participants_adult_count",
    "offer_variants_products_product_data_rateplan_code",
    "offer_variants_products_product_data_rateplan_title",
    "offer_variants_products_ref",
    "offer_variants_total_participants",
    "offer_variants_products_title",
    "offer_variants_products_to_date",
    "offer_variants_products_from_date",
    "offer_variants_products_pu_participants",
    "offer_variants_products_product_data_mealplan_code"
]
file = file.drop(columns=offer_drop_cols)

### Deal with missing

In [ ]:
# 1- Participant composition columns
participant_cols = [
    "products_participants_adult_count",
    "products_participants_child_count",
    "products_participants_teenager_count",
    "products_participants_infant_count",
    "products_pu_participants"
]

file[participant_cols] = file[participant_cols].fillna(0)


In [ ]:
# 2- Pricing detail columns
pricing_drop_cols = [
    "total_price_details_pu_price",
    "total_price_details_quantity",
    "total_price_details_amount",
    "total_price_details_title"
]
file = file.drop(columns=pricing_drop_cols)

In [ ]:
# 3- Advertising & campaign columns
file["advertising_source"] = file["advertising_source"].fillna("organic")
file["advertising_campaign"] = file["advertising_campaign"].fillna("organic")

In [ ]:
# 4- Product semantic
file["products_description"] = file["products_description"].fillna("No description available")

In [ ]:
# 5- Customer aggregated behavior
file["customer_total_revenue"] = file["customer_total_revenue"].fillna(0)
file["customer_stay_count"] = file["customer_stay_count"].fillna(1)

In [ ]:
# 6- Demographics & location
file["customer_gender"] = file["customer_gender"].fillna("unknown")
file["customer_address_city"] = file["customer_address_city"].fillna("unknown")
file["customer_address_country_iso"] = file["customer_address_country_iso"].fillna("unknown")

In [ ]:
# 7- Product rate & meal plan codes
rate_cols = [
    "products_product_data_rateplan_code",
    "products_product_data_rateplan_title",
    "products_product_data_mealplan_code"
]

file[rate_cols] = file[rate_cols].fillna("unknown")

In [ ]:
# 8- customer main
file["customer_main"] = file["customer_main"].fillna(1.0)

In [ ]:
# 9- customer_last_stay & customer_average_stay_time
# Case A — Customer has historical records
file["customer_average_stay_time"] = (
    file.groupby("customer_id")["customer_average_stay_time"]
      .transform(lambda x: x.fillna(x.mean()))
)

file["customer_last_stay"] = (
    file.groupby("customer_id")["customer_last_stay"]
      .transform(lambda x: x.fillna(x.max()))
)

In [ ]:
# Case B — No historical values exist
file["customer_average_stay_time"] = file["customer_average_stay_time"].fillna(file["customer_average_stay_time"].median())

In [ ]:
# 10- products_type, products_title
file["products_type"] = file["products_type"].fillna("unknown")
file["products_title"] = file["products_title"].fillna("unknown")

## Data Type Standardization

In [ ]:
df = file

def to_date_only(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce")
    return dt.apply(lambda x: x.date() if pd.notna(x) else pd.NA)

# ------------------------------------------------------------
# 1) Date/datetime columns -> keep ONLY date (drop time if present)
# ------------------------------------------------------------
date_cols = [
    "ordered_at",
    "products_from_date",
    "products_to_date",
    "customer_birthday",
    "customer_registration",
    "customer_last_contact",
    "customer_last_stay",
]

for c in date_cols:
    if c in df.columns:
        df[c] = to_date_only(df[c])

# ------------------------------------------------------------
# 2) Integer-like columns (counts / flags / ids) -> nullable Int64
#    (keeps missing values as <NA>)
# ------------------------------------------------------------
int_like_cols = [
    "customer_id",
    "customer_main",
    "customer_vip",
    "products_pu_participants",
    "products_participants_adult_count",
    "products_participants_child_count",
    "products_participants_teenager_count",
    "products_participants_infant_count",
    "customer_stay_count",
]

for c in int_like_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").round(0).astype("Int64")

# ------------------------------------------------------------
# 3) Numeric continuous / money columns -> float
# ------------------------------------------------------------
float_cols = [
    "total_amount",
    "customer_total_revenue",
    "customer_average_stay_time",
]

for c in float_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ------------------------------------------------------------
# 4) Text/categorical columns -> pandas string dtype (cleaner than object)
# ------------------------------------------------------------
text_cols = [
    "products_type",
    "products_title",
    "products_description",
    "products_product_data_mealplan_code",
    "products_product_data_rateplan_code",
    "products_product_data_rateplan_title",
    "customer_language",
    "customer_gender",
    "customer_address_city",
    "customer_address_country_iso",
    "advertising_source",
    "advertising_campaign",
]

for c in text_cols:
    if c in df.columns:
        df[c] = df[c].astype("string")


In [ ]:
file = df

## Second Copy

### Save a copy

In [ ]:
def save_dataframe(df, data_path='data.json', schema_path='schema.json'):
    # Save the data
    df.to_json(data_path, orient="records", indent=2, date_format='iso', force_ascii=False)
    
    # Save the schema (data types)
    schema = {col: str(dtype) for col, dtype in df.dtypes.items()}
    with open(schema_path, "w") as f:
        json.dump(schema, f, indent=2)

save_dataframe(file, data_path="Figure_out_data_withoutEdit.json", schema_path="Figure_out_data_schema_withoutEdit.json")

### Load the Copy

In [ ]:
def load_dataframe(data_path='data.json', schema_path='schema.json'):
    # Load data
    df = pd.read_json(data_path, orient="records")
    
    # Load schema
    with open(schema_path) as f:
        schema = json.load(f)
    
    # Identify column types
    datetime_cols = [col for col, dtype in schema.items() if 'datetime64' in dtype]
    category_cols = [col for col, dtype in schema.items() if dtype == 'category']
    
    # Reparse datetimes
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], utc='UTC' in schema[col])
    
    # Restore categories
    for col in category_cols:
        df[col] = df[col].astype('category')
    
    return df

file = load_dataframe(data_path="Figure_out_data_withoutEdit.json", schema_path="Figure_out_data_schema_withoutEdit.json")

## Feature Engineering

### Calculate the Age and Group Age

In [ ]:
# Step 1: Ensure date columns are datetime
file["ordered_at"] = pd.to_datetime(file["ordered_at"], errors="coerce")
file["customer_birthday"] = pd.to_datetime(file["customer_birthday"], errors="coerce")
# Step 2: Calculate customer age at time of order (in years)
file["customer_age"] = (
    (file["ordered_at"] - file["customer_birthday"]).dt.days // 365
)
# Step 3: Remove unrealistic ages
file.loc[
    (file["customer_age"] < 0) | (file["customer_age"] > 120),
    "customer_age"
] = pd.NA
file["customer_age"] = file["customer_age"].astype("Int64")
# Step 4: Define age groups
age_bins = [0, 17, 25, 40, 60, 120]
age_labels = ["<18", "18–25", "26–40", "41–60", "60+"]

file["customer_age_group"] = pd.cut(
    file["customer_age"],
    bins=age_bins,
    labels=age_labels
)

### Calculate the Booking lead time

In [ ]:
# Ensure date columns are datetime
file["ordered_at"] = pd.to_datetime(file["ordered_at"], errors="coerce")
file["products_from_date"] = pd.to_datetime(file["products_from_date"], errors="coerce")

# Initialize booking_lead_time with NA
file["booking_lead_time"] = pd.NA

# Calculate lead time in days where both dates exist
mask = file["ordered_at"].notna() & file["products_from_date"].notna()

file.loc[mask, "booking_lead_time"] = (
    (file.loc[mask, "products_from_date"] - file.loc[mask, "ordered_at"]).dt.days
)

# Remove invalid negative lead times (do NOT drop rows)
file.loc[file["booking_lead_time"] < 0, "booking_lead_time"] = pd.NA

# Convert to nullable integer (days)
file["booking_lead_time"] = file["booking_lead_time"].astype("Int64")

### Calculate the Occupancy

In [ ]:
# Ensure date columns are datetime
file["products_from_date"] = pd.to_datetime(file["products_from_date"], errors="coerce")
file["products_to_date"] = pd.to_datetime(file["products_to_date"], errors="coerce")

# Initialize occupancy_duration with NA
file["occupancy_duration"] = pd.NA

# Compute occupancy duration (in days) where both dates exist
mask = file["products_from_date"].notna() & file["products_to_date"].notna()

file.loc[mask, "occupancy_duration"] = (
    file.loc[mask, "products_to_date"] - file.loc[mask, "products_from_date"]
).dt.days

# Remove invalid durations (0 or negative) — do NOT drop rows
file.loc[file["occupancy_duration"] <= 0, "occupancy_duration"] = pd.NA

# Convert to nullable integer
file["occupancy_duration"] = file["occupancy_duration"].astype("Int64")

### Calculate the total orders

In [ ]:
order_counts = file["customer_id"].value_counts(dropna=False)

# Add/overwrite total_orders per customer_id
file["total_orders"] = file["customer_id"].map(order_counts).fillna(0).astype("Int64")


### Calculating Historical Annual CLV (Customer Lifetime Value)
Measures how much value a customer generated per year, based on past data.

We used a **customized Historical CLV** formula:

$$
\textbf{CLV} = \frac{\text{Average Order Value} \times \text{Total Orders}}{\text{Customer Lifespan (in years)}}
$$

---

#### Where:

* **Average Order Value (AOV)** =

  $$
  \frac{\text{Total Revenue}}{\text{Total Orders}}
  $$

* **Customer Lifespan (in years)** =

  $$
  \frac{\text{Last Order Date} - \text{First Order Date}}{365}
  $$

In [ ]:
# Ensure ordered_at is datetime
file["ordered_at"] = pd.to_datetime(file["ordered_at"], errors="coerce")

# (Optional safety) If you want to guarantee you only compute CLV on rows with a valid date:
orders = file[file["ordered_at"].notna()].copy()
# If your df is already orders-only, you can also do: orders = df.copy()

# Group by customer_id
clv_df = orders.groupby("customer_id").agg(
    total_revenue=("total_amount", "sum"),
    total_orders=("customer_id", "size"),        # counts rows/orders per customer
    first_order=("ordered_at", "min"),
    last_order=("ordered_at", "max"),
).reset_index()

# Calculate metrics
clv_df["avg_order_value"] = clv_df["total_revenue"] / clv_df["total_orders"]

clv_df["lifespan_years"] = (clv_df["last_order"] - clv_df["first_order"]).dt.days / 365
clv_df["lifespan_years"] = clv_df["lifespan_years"].fillna(0).clip(lower=1)  # avoid 0 years

# Final CLV
clv_df["CLV"] = clv_df["avg_order_value"] * clv_df["total_orders"] / clv_df["lifespan_years"]

# Round numeric values
cols_round = ["total_revenue", "avg_order_value", "lifespan_years", "CLV"]
clv_df[cols_round] = clv_df[cols_round].round(2)

# Merge back into the main df
file = file.merge(
    clv_df[["customer_id", "total_revenue", "avg_order_value", "lifespan_years", "CLV"]],
    on="customer_id",
    how="left"
)

### Dropping customer_birthday, products_from_date, products_to_date

In [ ]:
cpp_cols = [
    "customer_birthday",
    "products_from_date",
    "products_to_date"
]

file = file.drop(columns=cpp_cols)

### Handle missing customer_last_stay

In [ ]:
file['customer_last_stay'] = pd.to_datetime(
    file['customer_last_stay'], errors='coerce'
)

In [ ]:
reference_date = file['ordered_at'].max()

file['days_since_last_stay'] = (
    reference_date - file['customer_last_stay']
).dt.days

In [ ]:
file['has_previous_stay'] = file['customer_last_stay'].notna().astype(int)

In [ ]:
file['days_since_last_stay'] = file['days_since_last_stay'].fillna(
    file['days_since_last_stay'].max() + 1
)

In [ ]:
file.drop(columns=['customer_last_stay'], inplace=True)

### Handle the rest missing columns 

In [ ]:
# customer_age_group
file['customer_age_group'] = (
    file['customer_age_group']
    .cat.add_categories('Unknown')
    .fillna('Unknown')
)

In [ ]:
# occupancy_duration
file['occupancy_duration'] = file['occupancy_duration'].fillna(0)

In [ ]:
# booking_lead_time
file['booking_lead_time'] = file['booking_lead_time'].fillna(0)

### Dropping customer_age avoids imputation bias

In [ ]:
file.drop(columns=['customer_age'], inplace=True)

# Save Final Version of Preprocess Stage

In [ ]:
def save_dataframe(df, data_path='data.json', schema_path='schema.json'):
    # Save the data
    df.to_json(data_path, orient="records", indent=2, date_format='iso', force_ascii=False)
    
    # Save the schema (data types)
    schema = {col: str(dtype) for col, dtype in df.dtypes.items()}
    with open(schema_path, "w") as f:
        json.dump(schema, f, indent=2)

save_dataframe(file, data_path="Data_Preprocessing.json", schema_path="Data_Preprocessing_schema.json")